# 遊戲環境使用指南

這個notebook展示如何使用遊戲環境來訓練和測試強化學習模型。我們將使用PPO（Proximal Policy Optimization）算法來訓練一個玩Ms. Pacman的智能體。

## 環境設置

### 注意使用python3.10 
# 在安裝相關套件時先安裝GPU的pytorch跑比較快

首先，我們需要安裝必要的庫並設置基本參數。

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [19]:
from stable_baselines3 import PPO
from game_environment import GameEnvironment

# 玩家信息
player_name = 'name'  # 替換成你的暱稱(隨便你想取啥)
student_id = 2           # 替換成你的學號(一定要是正確的學號)

## 創建遊戲環境

我們可以通過GameEnvironment類來創建遊戲環境。這個環境提供了兩個重要的自定義選項：

1. `use_custom_reward`: 是否使用自定義獎勵函數
2. `use_custom_obs`: 是否使用自定義觀察處理

**注意**: 這些自定義選項只在訓練過程中使用，最終評分時會使用原始環境以確保公平性。

In [3]:
# 創建遊戲環境
env = GameEnvironment(
    env_id="ALE/MsPacman-v5",  # 不能動
    use_custom_reward=True,    # 只在訓練時使用自定義獎勵函數
    use_custom_obs=False       # 不使用自定義觀察處理
)

## 模型參數設置

PPO算法有多個可調整的參數，這些參數會影響模型的訓練效果：

- `learning_rate`: 學習率，控制每次更新的步長
- `n_steps`: 每次更新前收集的環境步數
- `batch_size`: 每次優化時使用的樣本數量
- `n_epochs`: 每次更新時重複訓練的次數
- `gamma`: 折扣因子，決定未來獎勵的重要性
- `gae_lambda`: GAE參數，用於平衡偏差和方差
- `clip_range`: PPO裁剪範圍，限制策略更新的幅度
- `ent_coef`: 熵係數，用於鼓勵探索

In [4]:
# 定義模型參數
model_params = {
    'verbose': 1,
    'learning_rate': 0.0003,
    'n_steps': 2048,
    'batch_size': 64,
    'n_epochs': 10,
    'gamma': 0.99,        
    'gae_lambda': 0.95,   
    'clip_range': 0.2,    
    'ent_coef': 0.01      
}

## 訓練方式示範

以下展示三種不同的訓練方式：
1. 使用默認設置訓練新模型
2. 使用自定義策略網絡訓練新模型
3. 繼續訓練已有的模型

In [5]:
# 1. 使用默認設置訓練新模型
env.train(
    model_class=PPO,
    model_params=model_params,
    total_timesteps=800,
    force_train=True      # 強制重新訓練
)

開始訓練新模型...
Using cuda device
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 498      |
|    ep_rew_mean     | 635      |
| time/              |          |
|    fps             | 229      |
|    iterations      | 1        |
|    time_elapsed    | 8        |
|    total_timesteps | 2048     |
---------------------------------
評估結果 - 平均獎勵: 415.00 (+/- 0.00)
模型已保存至: models/MsPacman-v5.zip


In [6]:
# 2. 使用自定義策略網絡訓練新模型
env.train(
    model_class=PPO,
    model_params=model_params,
    total_timesteps=800,
    force_train=True,
    use_custom_policy=True    # 使用自定義策略網絡
)

開始訓練新模型...
Using cuda device
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 467      |
|    ep_rew_mean     | 565      |
| time/              |          |
|    fps             | 185      |
|    iterations      | 1        |
|    time_elapsed    | 11       |
|    total_timesteps | 2048     |
---------------------------------
評估結果 - 平均獎勵: 415.00 (+/- 0.00)
模型已保存至: models/MsPacman-v5.zip


In [7]:
# 3. 繼續訓練已有的模型
env.train(
    model_class=PPO,
    model_params=model_params,
    total_timesteps=400,      # 額外訓練400步
    continue_from="models/MsPacman-v5.zip"  # 指定要繼續訓練的模型路徑
)

載入已存在的模型: models/MsPacman-v5.zip
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.
繼續訓練模型...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 502      |
|    ep_rew_mean     | 572      |
| time/              |          |
|    fps             | 151      |
|    iterations      | 1        |
|    time_elapsed    | 13       |
|    total_timesteps | 4096     |
---------------------------------
繼續訓練完成，評估模型中...
評估結果 - 平均獎勵: 115.00 (+/- 0.00)
更新後的模型已保存至: models/MsPacman-v5.zip


## 遊玩並提交結果

訓練完成後，我們可以使用訓練好的模型來玩遊戲並提交結果。

**注意**: 在這個階段，環境會使用原始設置（不使用自定義獎勵或觀察），以確保評分的公平性。

In [21]:
# 使用原始環境玩遊戲並提交結果
env.play(player_name, student_id, show_game=True, window_scale=3.0)

遊戲結束！總分：160.0
視頻成功保存到: recordings/starpig_gameplay.mp4
分數和影片提交成功！


## 自定義說明

你可以通過以下方式自定義強化學習模型：

### 1. 修改獎勵函數（僅影響訓練過程）
- 打開 `custom_wrappers.py`
- 在 `CustomRewardWrapper` 類中修改 `step` 方法
- 可以根據遊戲狀態設計新的獎勵計算方式

### 2. 修改觀察處理（僅影響訓練過程）
- 打開 `custom_wrappers.py`
- 在 `CustomObservationWrapper` 類中修改 `observation` 方法
- 可以對遊戲畫面進行預處理，如轉灰度、裁剪等

### 3. 修改策略網絡
- 打開 `custom_policy.py`
- 在 `CustomCNN` 類中修改網絡架構
- 可以調整卷積層、全連接層的參數和結構

### 4. 調整訓練參數
- 修改 `model_params` 字典中的參數
- 可以調整學習率、批次大小、訓練步數等

## 重要說明

1. 自定義獎勵和觀察處理只在訓練過程中使用
2. 最終遊玩和提交分數時會使用原始環境，以確保所有學生的評分標準一致
3. 這種設計允許學生通過自定義訓練過程來優化模型，同時保持評分的公平性

## 優化建議

1. 先使用默認設置運行遊戲，了解基本效果
2. 逐步調整單個組件（獎勵/觀察/網絡/參數）
3. 觀察修改後的訓練效果，進行進一步優化
4. 可以嘗試組合不同的修改來獲得最佳訓練效果
5. 記住最終評分是在原始環境中進行，所以要確保模型在原始環境中表現良好